In [4]:
pip install -r requirements.txt

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ------ --------------------------------- 1.8/11.3 MB 14.3 MB/s eta 0:00:01
   -------------------- ------------------- 5.8/11.3 MB 20.7 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 20.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------- ----------------------- 5.2/12.9 MB 35.6 MB/s eta 0:00:01
   ---------------------------------------- 12.9/12.9 MB 33.7 MB/s  0:00:00
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   -------------------------------------- - 8.7/8.9 MB 48.8 MB/s

In [5]:
import pandas as pd
import numpy as np
import os
import sys

sys.setrecursionlimit(2147483647)
from sklearn.model_selection import train_test_split

import glob

In [6]:
def add_gaussian_noise(df, noise_std=0.02):
    """
    Add Gaussian noise to Voltage column.
    noise_std: fixed std dev for noise.
    """
    if 'Voltage' not in df.columns:
        return df
    
    noise = np.random.normal(0, noise_std, size=len(df))
    df = df.copy()
    df['Voltage'] += noise
    return df

In [7]:
def process_sequences(input_dir='data/sequences_by_hour', output_dir='data/sequences_noisy', noise_std=0.02):
    """
    Add noise to all sequences and save noisy versions.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    sequence_files = glob.glob(f'{input_dir}/hour_*/v*.csv')
    print(f"Found {len(sequence_files)} sequences to process")
    
    for file_path in sequence_files:
        df = pd.read_csv(file_path)
        df = df.astype({
            'Time': float,
            'Voltage': float,
            'Peak': int,
            'Label': int
        })
        df_noisy = add_gaussian_noise(df, noise_std)
        
        # Keep only essential columns
        columns_to_keep = ['Time', 'Voltage', 'Peak', 'Label']
        df_noisy = df_noisy[columns_to_keep]
        
        # Output path
        rel_path = os.path.relpath(file_path, input_dir)
        output_path = os.path.join(output_dir, rel_path)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        df_noisy.to_csv(output_path, index=False)
        print(f"Processed {rel_path} → {output_path}")


In [8]:
def split_sequences(input_dir='data/sequences_noisy', output_dir='data/ml_splits', test_size=0.2, val_size=0.2):
    """
    Split all sequences into train/val/test sets.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    sequence_files = glob.glob(f'{input_dir}/hour_*/v*.csv')
    print(f"Splitting {len(sequence_files)} sequences")
    
    # Collect all data
    all_data = []
    for file_path in sequence_files:
        df = pd.read_csv(file_path)
        df = df.astype({
            'Time': float,
            'Voltage': float,
            'Peak': int,
            'Label': int
        })
        # Add sequence identifier
        seq_id = os.path.basename(file_path).replace('.csv', '')
        hour = os.path.basename(os.path.dirname(file_path))
        df['Sequence_ID'] = f"{hour}_{seq_id}"
        all_data.append(df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f"Total combined data: {len(combined_df)} rows")
    
    # Split by sequences to avoid data leakage
    sequence_ids = combined_df['Sequence_ID'].unique()
    print(f"Unique sequences: {len(sequence_ids)}")
    
    # First split: train + (val+test)
    train_seqs, temp_seqs = train_test_split(sequence_ids, test_size=(test_size + val_size), random_state=42)
    
    # Second split: val and test
    val_seqs, test_seqs = train_test_split(temp_seqs, test_size=test_size/(test_size + val_size), random_state=42)
    
    print(f"Train sequences: {len(train_seqs)}, Val: {len(val_seqs)}, Test: {len(test_seqs)}")
    
    # Filter data
    train_df = combined_df[combined_df['Sequence_ID'].isin(train_seqs)].drop(columns=['Sequence_ID'])
    val_df = combined_df[combined_df['Sequence_ID'].isin(val_seqs)].drop(columns=['Sequence_ID'])
    test_df = combined_df[combined_df['Sequence_ID'].isin(test_seqs)].drop(columns=['Sequence_ID'])
    
    # Save
    train_df.to_csv(f'{output_dir}/train.csv', index=False)
    val_df.to_csv(f'{output_dir}/val.csv', index=False)
    test_df.to_csv(f'{output_dir}/test.csv', index=False)
    
    print(f"Saved train.csv: {len(train_df)} rows")
    print(f"Saved val.csv: {len(val_df)} rows")
    print(f"Saved test.csv: {len(test_df)} rows")


In [9]:
print("Starting data augmentation and splitting...")

# Add noise
process_sequences(noise_std=0.02)

# Split
split_sequences()

print("✅ Done!")

Starting data augmentation and splitting...
Found 36 sequences to process
Processed hour_06\v1.csv → data/sequences_noisy\hour_06\v1.csv
Processed hour_06\v2.csv → data/sequences_noisy\hour_06\v2.csv
Processed hour_07\v1.csv → data/sequences_noisy\hour_07\v1.csv
Processed hour_07\v2.csv → data/sequences_noisy\hour_07\v2.csv
Processed hour_08\v1.csv → data/sequences_noisy\hour_08\v1.csv
Processed hour_08\v2.csv → data/sequences_noisy\hour_08\v2.csv
Processed hour_09\v1.csv → data/sequences_noisy\hour_09\v1.csv
Processed hour_09\v2.csv → data/sequences_noisy\hour_09\v2.csv
Processed hour_10\v1.csv → data/sequences_noisy\hour_10\v1.csv
Processed hour_10\v2.csv → data/sequences_noisy\hour_10\v2.csv
Processed hour_11\v1.csv → data/sequences_noisy\hour_11\v1.csv
Processed hour_11\v2.csv → data/sequences_noisy\hour_11\v2.csv
Processed hour_12\v1.csv → data/sequences_noisy\hour_12\v1.csv
Processed hour_12\v2.csv → data/sequences_noisy\hour_12\v2.csv
Processed hour_13\v1.csv → data/sequences_no

In [10]:


# List of files to analyze
files_to_check = [
    'data/ml_splits/train.csv',
    'data/ml_splits/val.csv',
    'data/ml_splits/test.csv',
    'output_label0_30min.csv'
]

print("Thống kê số lượng label trong từng file:")
print("=" * 50)

for file_path in files_to_check:
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        if 'Label' in df.columns:
            label_counts = df['Label'].value_counts().sort_index()
            print(f"\nFile: {file_path}")
            print(f"Tổng số mẫu: {len(df)}")
            for label in [0, 1, 2]:
                count = label_counts.get(label, 0)
                print(f"Label {label}: {count} ({count/len(df)*100:.1f}%)")
        else:
            print(f"\nFile: {file_path} - Không có cột 'Label'")
    else:
        print(f"\nFile: {file_path} - Không tồn tại")

print("\n✅ Hoàn thành thống kê!")

Thống kê số lượng label trong từng file:

File: data/ml_splits/train.csv
Tổng số mẫu: 19353915
Label 0: 11967561 (61.8%)
Label 1: 3825896 (19.8%)
Label 2: 3560458 (18.4%)

File: data/ml_splits/val.csv
Tổng số mẫu: 6451305
Label 0: 3812455 (59.1%)
Label 1: 1451890 (22.5%)
Label 2: 1186960 (18.4%)

File: data/ml_splits/test.csv
Tổng số mẫu: 7372920
Label 0: 4999298 (67.8%)
Label 1: 1186677 (16.1%)
Label 2: 1186945 (16.1%)

File: output_label0_30min.csv
Tổng số mẫu: 460808
Label 0: 460808 (100.0%)
Label 1: 0 (0.0%)
Label 2: 0 (0.0%)

✅ Hoàn thành thống kê!
